In [ ]:
#Lab 2.4 — Normalization: BatchNorm vs LayerNorm
#1 CNN with/without BatchNorm
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_tfm = transforms.ToTensor()
train_full = datasets.FashionMNIST('data', train=True, download=True, transform=base_tfm)
test = datasets.FashionMNIST('data', train=False, download=True, transform=base_tfm)
n_val = 5000
train, val = random_split(train_full, [len(train_full)-n_val, n_val], generator=torch.Generator().manual_seed(42))
def loaders(dataset, bs=128):
    return (DataLoader(dataset, batch_size=bs, shuffle=True),
            DataLoader(val, batch_size=bs),
            DataLoader(test, batch_size=bs))
train_dl, val_dl, test_dl = loaders(train)

@torch.no_grad()
def evaluate(model, dl):
    model.eval(); tot=0; correct=0
    for x,y in dl:
        x,y = x.to(device), y.to(device)
        logits = model(x); loss = F.cross_entropy(logits, y)
        tot += loss.item()*x.size(0)
        correct += (logits.argmax(1)==y).sum().item()
    return tot/len(dl.dataset), correct/len(dl.dataset)

class CNN_NoNorm(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(64*7*7,128), nn.ReLU(), nn.Linear(128,10)
        )
    def forward(self,x): return self.net(x)

class CNN_BN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(64*7*7,128), nn.BatchNorm1d(128), nn.ReLU(), nn.Linear(128,10)
        )
    def forward(self,x): return self.net(x)

#2 MLP with/without LayerNorm
class MLP_NoNorm(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1, self.fc2, self.out = nn.Linear(28*28,256), nn.Linear(256,128), nn.Linear(128,10)
    def forward(self,x):
        x = x.view(x.size(0),-1)
        x = F.relu(self.fc1(x)); x = F.relu(self.fc2(x)); return self.out(x)

class MLP_LN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1, self.ln1 = nn.Linear(28*28,256), nn.LayerNorm(256)
        self.fc2, self.ln2 = nn.Linear(256,128), nn.LayerNorm(128)
        self.out = nn.Linear(128,10)
    def forward(self,x):
        x = x.view(x.size(0),-1)
        x = F.relu(self.ln1(self.fc1(x)))
        x = F.relu(self.ln2(self.fc2(x)))
        return self.out(x)

#3 Generic trainer for quick comparisons
def quick_train(model, epochs=6, lr=1e-3):
    model = model.to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    tr_hist, val_hist = [], []
    for ep in range(epochs):
        model.train(); tot=0
        for x,y in train_dl:
            x,y = x.to(device), y.to(device)
            opt.zero_grad(); loss = F.cross_entropy(model(x), y)
            loss.backward(); opt.step(); tot += loss.item()*x.size(0)
        tr = tot/len(train_dl.dataset); vl, acc = evaluate(model, val_dl)
        tr_hist.append(tr); val_hist.append(vl)
        print(f"{type(model).__name__:12s} ep{ep+1}: train {tr:.3f} val {vl:.3f} acc {acc:.3f}")
    return model, tr_hist, val_hist

#4 Run comparisons
cnn0, t0, v0 = quick_train(CNN_NoNorm(), epochs=6)
cnnBN, t1, v1 = quick_train(CNN_BN(),     epochs=6)
mlp0, tm0, vm0 = quick_train(MLP_NoNorm(),  epochs=6)
mlpLN, tm1, vm1 = quick_train(MLP_LN(),     epochs=6)

#5 (Optional) Plot loss curves
def plot_curves(curves, title):
    plt.figure()
    for name, hist in curves:
        plt.plot(hist, label=name)
    plt.title(title); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.show()

plot_curves([('CNN no norm (val)', v0), ('CNN BN (val)', v1)], "CNN: Validation Loss")
plot_curves([('MLP no norm (val)', vm0), ('MLP LN (val)', vm1)], "MLP: Validation Loss")

#6 Final test accuracy
for name, m in [('CNN_NoNorm', cnn0), ('CNN_BN', cnnBN), ('MLP_NoNorm', mlp0), ('MLP_LN', mlpLN)]:
    _, acc = evaluate(m, test_dl)
    print(f"{name:12s} | test acc: {acc:.3f}")

CNN_NoNorm   ep1: train 0.536 val 0.371 acc 0.868
